In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Feature Distribution Visualization Utilities (`plots/plot_utils.ipynb`)

This notebook generates comprehensive visualization plots for:
1. **Configured Features**: Features specified in `config/triage_conf.json`.
2. **All Features in Dataset**: Every variable present inside the complete `datasets/5v_cleandf.RData` dataset.

### Visualizations Generated:
- **Target Class Distribution (`esi`)**: Bar chart of patient counts across ESI triage levels.
- **Categorical Feature Distribution (`gender`)**: Stacked & grouped bar charts across ESI levels.
- **Configured Feature Distributions**: Density KDE plots and boxplots by ESI level.
- **All Dataset Features Distribution**: Full faceted grid density plots and boxplots covering **every column inside `.RData`**.
- **Saved Artifacts**: PNG plots exported to `plots/image/` and summary tables exported to `plots/csv/`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(ggplot2)
library(dplyr)
library(tidyr)
library(gridExtra)

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

config <- fromJSON(config_path)

cat("=== Configuration Loaded ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Features Count:  ", length(config$features$data_name), "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Dataset & Ensure Image/CSV Output Directories
# ---------------------------------------------------------
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]

raw_df <- get(data_obj_name, envir = data_env)

target_col   <- config$classes$target_col
feature_cols <- config$features$data_name
target_classes <- as.character(config$classes$outputs)

selected_cols <- intersect(c(feature_cols, target_col), names(raw_df))
df <- raw_df[, selected_cols, drop = FALSE]

df[[target_col]] <- factor(df[[target_col]], levels = target_classes)

# Ensure output directories exist
img_dir <- "../plots/image"
if (!dir.exists(img_dir)) img_dir <- "image"
if (!dir.exists(img_dir)) dir.create(img_dir, recursive = TRUE)

csv_dir <- "../plots/csv"
if (!dir.exists(csv_dir)) csv_dir <- "csv"
if (!dir.exists(csv_dir)) dir.create(csv_dir, recursive = TRUE)

cat(sprintf("Loaded dataset object '%s': %d rows x %d cols (JSON subset: %d cols)\n", 
            data_obj_name, nrow(raw_df), ncol(raw_df), ncol(df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Target Class (esi) Distribution Plot
# ---------------------------------------------------------
target_summary <- df %>%
  group_by(.data[[target_col]]) %>%
  summarise(Count = n(), .groups = "drop") %>%
  mutate(Percentage = (Count / sum(Count)) * 100)

cat("=== ESI Target Class Summary ===\n")
print(target_summary)
write.csv(target_summary, file.path(csv_dir, "esi_target_distribution.csv"), row.names = FALSE)

p_target <- ggplot(target_summary, aes(x = .data[[target_col]], y = Count, fill = .data[[target_col]])) +
  geom_bar(stat = "identity", color = "black", alpha = 0.85) +
  geom_text(aes(label = sprintf("%s\n(%.1f%%)", format(Count, big.mark=","), Percentage)), vjust = -0.2, size = 3.8) +
  scale_fill_brewer(palette = "Set1") +
  theme_minimal(base_size = 13) +
  labs(
    title = "Target Distribution: Emergency Severity Index (ESI)",
    subtitle = "Patient frequency across ESI triage levels 1 to 5",
    x = "ESI Triage Level",
    y = "Patient Count",
    fill = "ESI Level"
  ) +
  theme(legend.position = "none", panel.grid.minor = element_blank())

print(p_target)
ggsave(file.path(img_dir, "target_esi_distribution.png"), plot = p_target, width = 8, height = 5)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Categorical Feature (gender) Distribution Plot
# ---------------------------------------------------------
if ("gender" %in% names(df)) {
  gender_summary <- df %>%
    group_by(gender, .data[[target_col]]) %>%
    summarise(Count = n(), .groups = "drop")
  
  write.csv(gender_summary, file.path(csv_dir, "gender_esi_distribution.csv"), row.names = FALSE)
  
  p_gender <- ggplot(df, aes(x = factor(gender), fill = .data[[target_col]])) +
    geom_bar(position = "dodge", color = "black", alpha = 0.85) +
    scale_fill_brewer(palette = "Set1") +
    theme_minimal(base_size = 13) +
    labs(
      title = "Gender Distribution by ESI Triage Level",
      subtitle = "Comparison of patient counts by gender across ESI categories",
      x = "Gender",
      y = "Patient Count",
      fill = "ESI Level"
    ) +
    theme(panel.grid.minor = element_blank())
  
  print(p_gender)
  ggsave(file.path(img_dir, "gender_distribution.png"), plot = p_gender, width = 8, height = 5)
}

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Continuous Numerical Feature Density & Boxplot Distributions
# ---------------------------------------------------------
num_cols <- names(df)[sapply(df, is.numeric)]

cat("Generating distribution plots for JSON numerical features:", paste(num_cols, collapse = ", "), "\n")

# Summary statistics for numerical features
num_summary <- df %>%
  pivot_longer(cols = all_of(num_cols), names_to = "Feature", values_to = "Value") %>%
  group_by(Feature) %>%
  summarise(
    Mean = round(mean(Value, na.rm = TRUE), 2),
    SD = round(sd(Value, na.rm = TRUE), 2),
    Median = round(median(Value, na.rm = TRUE), 2),
    IQR = round(IQR(Value, na.rm = TRUE), 2),
    Min = round(min(Value, na.rm = TRUE), 2),
    Max = round(max(Value, na.rm = TRUE), 2),
    .groups = "drop"
  )

print(num_summary)
write.csv(num_summary, file.path(csv_dir, "numeric_features_summary.csv"), row.names = FALSE)

# Long format for faceted plotting
df_long <- df %>%
  pivot_longer(cols = all_of(num_cols), names_to = "Feature", values_to = "Value")

# 1. Density Histograms Faceted Grid
p_density <- ggplot(df_long, aes(x = Value, fill = .data[[target_col]])) +
  geom_density(alpha = 0.45) +
  facet_wrap(~ Feature, scales = "free", ncol = 3) +
  scale_fill_brewer(palette = "Set1") +
  theme_minimal(base_size = 12) +
  labs(
    title = "Feature Density Distributions Grouped by ESI Class",
    subtitle = "KDE plots for all numerical vital signs and demographics",
    x = "Feature Value",
    y = "Density",
    fill = "ESI Level"
  ) +
  theme(legend.position = "bottom", panel.grid.minor = element_blank())

print(p_density)
ggsave(file.path(img_dir, "numeric_features_density.png"), plot = p_density, width = 12, height = 8)

# 2. Boxplots by ESI Triage Level
p_box <- ggplot(df_long, aes(x = .data[[target_col]], y = Value, fill = .data[[target_col]])) +
  geom_boxplot(outlier.size = 0.5, outlier.alpha = 0.3, alpha = 0.8) +
  facet_wrap(~ Feature, scales = "free_y", ncol = 3) +
  scale_fill_brewer(palette = "Set1") +
  theme_minimal(base_size = 12) +
  labs(
    title = "Numerical Feature Distributions across ESI Triage Levels",
    subtitle = "Boxplots highlighting medians, interquartile ranges, and outliers",
    x = "ESI Level",
    y = "Feature Value",
    fill = "ESI Level"
  ) +
  theme(legend.position = "none", panel.grid.minor = element_blank())

print(p_box)
ggsave(file.path(img_dir, "numeric_features_boxplots.png"), plot = p_box, width = 12, height = 8)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Summary Report of Generated Image Artifacts
# ---------------------------------------------------------
cat("=== All Feature Distribution Plotting Complete ===\n")
cat("Generated image plots saved to:\n")
cat("  - Target Distribution:     ", file.path(img_dir, "target_esi_distribution.png"), "\n")
cat("  - Gender Distribution:     ", file.path(img_dir, "gender_distribution.png"), "\n")
cat("  - JSON Numerical Density:  ", file.path(img_dir, "numeric_features_density.png"), "\n")
cat("  - JSON Numerical Boxplots: ", file.path(img_dir, "numeric_features_boxplots.png"), "\n")
cat("Generated CSV summaries saved to:\n")
cat("  - ESI Target CSV:          ", file.path(csv_dir, "esi_target_distribution.csv"), "\n")
cat("  - JSON Numeric Summary CSV: ", file.path(csv_dir, "numeric_features_summary.csv"), "\n")